# Budget analysis

Analyze spending from bank CSV/Excel exports.

1. Drop your exports into `data/raw/` (git-ignored — never committed).
2. Optionally copy `config/categories.example.yaml` → `config/categories.yaml` and customize keywords.
3. Run all cells. If `data/raw/` is empty, sample data is used.

Convention: **negative = money out**, **positive = money in**.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import load_all_raw, load_transactions, categorize, load_non_spending

NON_SPENDING = load_non_spending()
RAW = ROOT / "data" / "raw"

try:
    accounts = {}
    for p in sorted(RAW.glob("*")):
        if p.suffix.lower() in {".csv", ".xlsx", ".xls"}:
            d = categorize(load_transactions(p))
            d["month"] = d["date"].dt.to_period("M").astype(str)
            accounts[p.stem] = d
    if not accounts:
        raise FileNotFoundError
    print("Loaded:", ", ".join(f"{k} ({len(v)} txns)" for k, v in accounts.items()))
except FileNotFoundError:
    d = categorize(load_transactions(ROOT / "data" / "sample_transactions.csv", account="sample"))
    d["month"] = d["date"].dt.to_period("M").astype(str)
    accounts = {"sample": d}
    print(f"No files in data/raw/ — using sample data ({len(d)} transactions).")

In [ ]:
def spending(df):
    s = df[(df["amount"] < 0) & (~df["category"].isin(NON_SPENDING))].copy()
    s["out"] = -s["amount"]
    return s

def income(df):
    return df[(df["amount"] > 0) & (~df["category"].isin(NON_SPENDING))]

rows = []
for name, df in accounts.items():
    inc, spe = income(df)["amount"].sum(), spending(df)["out"].sum()
    rows.append({
        "account": name,
        "from": df["date"].min().date(),
        "to": df["date"].max().date(),
        "income": round(inc, 2),
        "spending": round(spe, 2),
        "net": round(inc - spe, 2),
    })
pd.DataFrame(rows).set_index("account")

## Per-account breakdown

Run the cells below for each account. Transfers and credit-card payments are excluded from spending.

In [ ]:
def report(name):
    df = accounts[name]
    spe, inc = spending(df), income(df)
    print(f"=== {name} ===")
    print(f"{df['date'].min().date()} -> {df['date'].max().date()}  ({len(df)} txns)")
    print(f"income:   ${inc['amount'].sum():>12,.2f}")
    print(f"spending: ${spe['out'].sum():>12,.2f}")

    monthly = pd.DataFrame({
        "income": inc.groupby("month")["amount"].sum(),
        "spending": spe.groupby("month")["out"].sum(),
    }).fillna(0).sort_index()
    by_cat = spe.groupby("category")["out"].sum().sort_values()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    monthly.plot(kind="bar", ax=axes[0])
    axes[0].set_title(f"{name}: income vs spending"); axes[0].set_ylabel("$")
    by_cat.plot(kind="barh", ax=axes[1])
    axes[1].set_title(f"{name}: by category"); axes[1].set_xlabel("$")
    plt.tight_layout(); plt.show()

for name in accounts:
    report(name)

In [ ]:
for name, df in accounts.items():
    spe = spending(df)
    unc = spe[spe["category"] == "Uncategorized"]
    print(f"--- {name}: uncategorized (${unc['out'].sum():,.2f}, {len(unc)} txns) ---")
    if len(unc):
        print((unc.groupby("description")["out"].sum().sort_values(ascending=False).head(10)).round(2).to_string())
    print()